# Distilling the data-analyst agent

LoRA fine-tunes a small student on the teacher's trajectories, then evaluates teacher vs zero-shot student vs distilled student on the held-out DABench subset. All logic lives in the repo's `distill/` and `eval/` modules; this notebook only clones, installs, and runs them.

Prerequisites: GPU accelerator on (T4), Internet on, and the teacher trajectories uploaded as a Kaggle Dataset.

In [ ]:
%cd /kaggle/working
!rm -rf data-analyst-agent
!git clone https://github.com/wahyuhiddayat/data-analyst-agent.git
%cd data-analyst-agent
!pip install -q -e . -r requirements-train.txt
# Kaggle ships an old torchao that PEFT rejects on import; it is unused here.
!pip uninstall -y -q torchao

## Prepare data and train

The `--input` path points to the uploaded trajectories dataset; adjust it if the dataset slug differs.

In [ ]:
!python -m distill.format \
    --input /kaggle/input/datasets/wahyuuhidaayat/deepseek-v4-flash/deepseek-v4-flash.jsonl \
    --output data/train.jsonl

In [ ]:
# max-length is capped for T4 memory: the fp32 loss over Qwen's 152K-vocab logits
# is the peak allocation. expandable_segments limits allocator fragmentation.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m distill.train \
    --model Qwen/Qwen2.5-3B-Instruct \
    --data data/train.jsonl \
    --out /kaggle/working/adapter \
    --max-length 4096 \
    --lora-rank 32

In [ ]:
# Zip the adapter right after training so it survives a later-cell failure.
!cd /kaggle/working && zip -r -q adapter.zip adapter && ls -lh adapter.zip

## Serve and evaluate

Start a local vLLM server exposing both the base model (zero-shot student) and the LoRA adapter (distilled student), then run the same evaluation harness against each. This cell is the one to watch on the first run: vLLM tool-calling on a T4 is the least-tested link.

In [ ]:
!pip install -q vllm
import subprocess, time, urllib.request

# Stop any server left over from a previous run so the port is free.
subprocess.run(["pkill", "-f", "vllm serve"])
time.sleep(5)

server = subprocess.Popen([
    "vllm", "serve", "Qwen/Qwen2.5-3B-Instruct",
    "--enable-lora", "--lora-modules", "distilled=/kaggle/working/adapter",
    "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
    "--max-model-len", "16384", "--port", "8000",
])

for _ in range(180):
    try:
        urllib.request.urlopen("http://localhost:8000/health")
        print("server ready")
        break
    except Exception:
        time.sleep(5)
else:
    raise RuntimeError("vLLM server did not become ready")

In [ ]:
import os
os.environ["DEEPSEEK_BASE_URL"] = "http://localhost:8000/v1"
os.environ["DEEPSEEK_API_KEY"] = "local"

!python -m eval.run_eval --model Qwen/Qwen2.5-3B-Instruct --config zeroshot --n 25 --seed 0
!python -m eval.run_eval --model distilled --config distilled --n 25 --seed 0

In [ ]:
print(open("eval/results/summary.md").read())

The summary now holds the four-way comparison (flash teacher, pro teacher, zero-shot student, distilled student). The adapter is in `/kaggle/working/adapter`, zipped as `adapter.zip`; Save Version is required for either to persist beyond the session.